# BÀI THỰC HÀNH: LÀM SẠCH DỮ LIỆU ĐỘC BẢN (CÁ NHÂN)
**Môn học:** Khai phá dữ liệu (Data Mining)  
**Sinh viên:** Nguyễn Trung Hậu  
**MSSV:** 2251010027  
**Lớp/Thực hành:** BTH02  
**Môi trường:** Anaconda / Jupyter Notebook (Python 3)

---
### Mục tiêu bài thực hành:
Thực hiện bước **Làm sạch dữ liệu (Data Cleaning)** trong quy trình KDD (Knowledge Discovery in Databases):
1. Loại bỏ các bản ghi trùng lặp ngẫu nhiên (`.drop_duplicates()`).
2. Điền khuyết dữ liệu số bằng phương pháp thích hợp (`Mean` cho Điểm số, `Median` cho Tuổi chứa ngoại lai).
3. Chuẩn hóa chuỗi văn bản (loại bỏ khoảng trắng thừa `.str.strip()`, đồng nhất tên tỉnh/thành phố `.replace()`).
4. Lọc bỏ các bản ghi vi phạm tính logic của dữ liệu (Tuổi ngoài 18–30, Điểm ngoài 0–10).

## Bước 1: Khởi tạo và tạo tệp dữ liệu thô chứa lỗi (Data Generation)
Khởi tạo seed ngẫu nhiên theo MSSV `2251010027` để tạo bộ dữ liệu 120 dòng kèm 6 dòng trùng lặp và các lỗi theo đặc tả.

In [1]:
import pandas as pd  # Thư viện chuyên thao tác với bảng dữ liệu DataFrame
import numpy as np   # Thư viện tính toán số học và sinh số ngẫu nhiên

MSSV = 2251010027    # Mã số sinh viên cá nhân dùng làm seed
np.random.seed(MSSV) # Cố định seed ngẫu nhiên: đảm bảo kết quả nhất quán mỗi lần chạy
n = 120              # Số lượng mẫu sinh viên ban đầu

# --- 1. Tạo danh sách MSSV ---
# List comprehension: tạo 120 mã sinh viên giảm dần từ MSSV gốc
mssv_list = [f"SV{MSSV - i}" for i in range(n)]

# --- 2. Tạo họ tên (Cố tình tạo lỗi khoảng trắng thừa ở 2 đầu) ---
ho = ["Nguyen", "Tran", "Le", "Pham", "Hoang", "Huynh", "Phan", "Vu", "Vo", "Dang"]
ten_dem = ["Van", "Thi", "Minh", "Hoang", "Duc", "Thu", "Anh", "Quoc", ""]
ten = ["An", "Binh", "Cuong", "Dung", "Em", "Phong", "Giang", "Hai", "Khanh", "Lan",
       "Mai", "Nam", "Oanh", "Phuc", "Quyen", "Son", "Tam", "Uyen", "Vinh", "Yen"]

ho_ten_list = []
for i in range(n):
    h = np.random.choice(ho)       # Chọn ngẫu nhiên 1 họ
    td = np.random.choice(ten_dem) # Chọn ngẫu nhiên 1 tên đệm
    t = np.random.choice(ten)      # Chọn ngẫu nhiên 1 tên chính
    name = f"{h} {td} {t}".replace("  ", " ").strip() # Gộp khoảng trắng kép thành đơn
    
    # np.random.rand() < 0.40: Xác suất 40% cố tình chèn khoảng trắng thừa ở 2 đầu chuỗi
    if np.random.rand() < 0.40:
        name = "  " + name + "  "
    ho_ten_list.append(name)

# --- 3. Tạo tuổi (Cố tình tạo lỗi: khuyết NaN, tuổi âm, tuổi quá lớn) ---
# np.random.randint(18, 31): Sinh số nguyên trong khoảng [18, 30]. .astype(float): ép kiểu thực để chứa NaN
tuoi_list = np.random.randint(18, 31, size=n).astype(float)

# Lấy ngẫu nhiên khoảng 8% số dòng không trùng lặp (replace=False) để gán thành NaN
nan_idx = np.random.choice(n, size=max(1, int(n * 0.08)), replace=False)
tuoi_list[nan_idx] = np.nan

# Cố tình gán 2 giá trị ngoại lai cực đoan: -12 (tuổi âm) và 195 (tuổi quá lớn)
outlier_idx = np.random.choice([i for i in range(n) if i not in nan_idx], size=2, replace=False)
tuoi_list[outlier_idx[0]] = -12
tuoi_list[outlier_idx[1]] = 195

# --- 4. Tạo điểm số (Cố tình tạo lỗi: khuyết NaN, điểm âm, vượt thang điểm 10) ---
# np.random.uniform(0, 10): Phân phối đều từ 0 đến 10, làm tròn 1 chữ số thập phân
diem_list = np.round(np.random.uniform(0, 10, size=n), 1)

# Lấy ngẫu nhiên khoảng 7% số dòng để gán thành NaN
nan_idx2 = np.random.choice(n, size=max(1, int(n * 0.07)), replace=False)
diem_list[nan_idx2] = np.nan

# Cố tình gán 2 giá trị vi phạm logic thang điểm: -2.0 (điểm âm) và 95.0 (vượt thang điểm 10)
err_idx = np.random.choice([i for i in range(n) if i not in nan_idx2], size=2, replace=False)
diem_list[err_idx[0]] = -2.0
diem_list[err_idx[1]] = 95.0

# --- 5. Tạo tỉnh/thành (Cố tình tạo lỗi định dạng không nhất quán) ---
tinh_variants = ["Ha Noi", "HN", "TP.HCM", "HCM", "Da Nang", "Can Tho", "Hai Phong"]
tinh_list = [np.random.choice(tinh_variants) for _ in range(n)]

# --- 6. Tạo DataFrame tổng hợp ---
# pd.DataFrame(): Đóng gói các mảng dữ liệu thành bảng 2 chiều
df = pd.DataFrame({
    "MSSV": mssv_list,
    "Ho_Ten": ho_ten_list,
    "Tuoi": tuoi_list,
    "Diem_So": diem_list,
    "Tinh_Thanh": tinh_list
})

# --- 7. Nhân bản 6 bản ghi ở cuối (Cố tình tạo lỗi dữ liệu trùng lặp) ---
duplicates = df.sample(n=6, random_state=MSSV) # Lấy ngẫu nhiên 6 dòng mẫu
df = pd.concat([df, duplicates], ignore_index=True) # Nối vào cuối bảng, đánh lại index từ 0 -> 125

# --- 8. Xuất file dữ liệu thô chứa lỗi ---
raw_filename = f"data_loi_{MSSV}.csv"
df.to_csv(raw_filename, index=False, encoding="utf-8") # index=False: không lưu cột số thứ tự mặc định

print(f"Đã tạo file: {raw_filename}")
print(f"Số dòng: {len(df)} (gồm 120 dòng gốc + 6 dòng trùng lặp)")
print(f"Số giá trị khuyết Tuoi: {df['Tuoi'].isna().sum()}")
print(f"Số giá trị khuyết Diem_So: {df['Diem_So'].isna().sum()}")
df.head(10)

## Bước 2: Đọc dữ liệu và đánh giá cấu trúc tổng thể (Data Profiling)
Rà soát cấu trúc bảng dữ liệu với `.info()`, thống kê mô tả `.describe()` để phát hiện các bất thường (min/max, ô thiếu) và hiển thị 5 dòng đầu bằng `.head()`.

In [2]:
# pd.read_csv(): Đọc tệp CSV thô vừa tạo vào DataFrame
df = pd.read_csv(f"data_loi_{MSSV}.csv", encoding="utf-8")

print("=== Thông tin cấu trúc dữ liệu (df.info()) ===")
# .info(): Xem số dòng, số cột, số lượng giá trị không rỗng (Non-Null) và kiểu dữ liệu (Dtype)
df.info()

print("\n=== Thống kê mô tả (df.describe()) ===") 
# .describe(): Thống kê số lượng (count), trung bình (mean), độ lệch chuẩn (std), min, max, tứ phân vị (25%, 50%, 75%)
# Dễ dàng phát hiện ngoại lai cực đoan: min Tuoi = -12, max Tuoi = 195; min Diem = -2, max Diem = 95
display(df.describe())

print("\n=== 5 dòng đầu tiên (df.head()) ===")
display(df.head())

## Bước 3: Xử lý bản ghi trùng lặp (Duplicate Records)
- Hàm `.duplicated().sum()`: Đếm số dòng bị nhân bản hoàn toàn.
- Hàm `.drop_duplicates(inplace=True)`: Xóa bỏ các dòng trùng lặp, giữ lại bản ghi đầu tiên.

In [3]:
# df.duplicated(): Trả về Series True/False đối với các dòng bị lặp lại
num_duplicates = df.duplicated().sum()
print(f"Số dòng trùng lặp phát hiện được: {num_duplicates}")

# df.drop_duplicates(inplace=True): Xóa bỏ dòng trùng trực tiếp trên bảng dữ liệu hiện tại
df.drop_duplicates(inplace=True)
print(f"Số dòng còn lại sau khi loại bỏ trùng: {len(df)}")

## Bước 4: Xử lý các giá trị khuyết thiếu (Missing Values)
- **Cột Diem_So:** Điền bằng giá trị **Trung bình (Mean)**.
- **Cột Tuoi:** Điền bằng giá trị **Trung vị (Median)** vì trung vị không bị méo mó bởi ngoại lai cực đoan (-12, 195).

In [4]:
# Đếm số giá trị khuyết TRƯỚC khi điền (dùng để trả lời Câu hỏi thu hoạch 1)
missing_tuoi = int(df["Tuoi"].isna().sum())
missing_diem = int(df["Diem_So"].isna().sum())
print(f"Số giá trị khuyết cột Tuoi: {missing_tuoi}")
print(f"Số giá trị khuyết cột Diem_So: {missing_diem}")

# .mean(): Tính trung bình cộng của cột điểm
diem_mean = df["Diem_So"].mean()
# .median(): Tính trung vị (vị trí 50%) của cột tuổi - mang tính bền vững (Robust) trước ngoại lai
tuoi_median = df["Tuoi"].median()

# .fillna(): Lấp đầy các ô NaN bằng giá trị chỉ định
df["Diem_So"] = df["Diem_So"].fillna(diem_mean)
df["Tuoi"] = df["Tuoi"].fillna(tuoi_median)

print("Đã điền xong giá trị khuyết.")
print(f"Kiểm tra - Tuoi còn khuyết: {df['Tuoi'].isna().sum()}")
print(f"Kiểm tra - Diem_So còn khuyết: {df['Diem_So'].isna().sum()}")

## Bước 5: Chuẩn hóa dữ liệu dạng văn bản (Text Standardization)
- `.str.strip()`: Xóa sạch khoảng trắng thừa ở hai đầu chuỗi họ tên.
- `.replace()`: Ánh xạ chuẩn hóa các biến thể viết tắt (`HN` -> `Ha Noi`, `HCM` -> `TP.HCM`).

In [5]:
# .str.strip(): Áp dụng hàm cắt khoảng trắng đầu và cuối chuỗi cho từng dòng trong cột họ tên
df["Ho_Ten"] = df["Ho_Ten"].str.strip()

# .replace(dict): Đồng nhất các tên viết tắt về một quy chuẩn chung
df["Tinh_Thanh"] = df["Tinh_Thanh"].replace({
    "HN": "Ha Noi",
    "HCM": "TP.HCM"
})

# .unique(): Liệt kê các giá trị duy nhất để kiểm tra tính nhất quán
print("Các giá trị độc nhất của cột Tinh_Thanh sau chuẩn hóa:")
print(df["Tinh_Thanh"].unique())

## Bước 6: Xử lý dữ liệu sai lỗi logic và xuất tệp tin sạch (Data Filtering)
- Lọc bỏ triệt để các bản ghi vi phạm logic: `18 <= Tuoi <= 30` và `0 <= Diem_So <= 10`.
- Xuất kết quả sạch ra tệp `{MSSV}_clean.csv`.

In [6]:
# Lọc dữ liệu theo điều kiện logic (Tuổi 18-30, Điểm 0-10)
# Toán tử '&': Phép AND giữa các điều kiện logic
# .copy(): Tạo bản sao mới độc lập trên RAM để tránh cảnh báo phân mảnh bộ nhớ
df_clean = df[
    (df["Tuoi"] >= 18) & (df["Tuoi"] <= 30) &
    (df["Diem_So"] >= 0) & (df["Diem_So"] <= 10)
].copy()

print(f"Số dòng còn lại sau khi lọc logic: {len(df_clean)}")

# Xuất tệp dữ liệu sạch sẵn sàng cho các mô hình Khai phá dữ liệu
clean_filename = f"{MSSV}_clean.csv"
df_clean.to_csv(clean_filename, index=False)
print(f"Đã xuất tệp: {clean_filename}")

# Kiểm tra lại chất lượng dữ liệu sạch
print("\n=== Kiểm tra dữ liệu sạch ===") 
print(df_clean.info())
print(f"Min Tuoi = {df_clean['Tuoi'].min()}, Max Tuoi = {df_clean['Tuoi'].max()}")
print(f"Min Diem_So = {df_clean['Diem_So'].min()}, Max Diem_So = {df_clean['Diem_So'].max()}")
display(df_clean.head())

---
## PHẦN VIẾT THU HOẠCH BẮT BUỘC

### **Câu hỏi 1:** Đối với MSSV của em (`2251010027`), hệ thống quét được bao nhiêu dòng trùng lặp và bao nhiêu ô khuyết ở cột Tuoi, Diem_So?
**Trả lời:**
- **Số dòng trùng lặp quét được:** `6` dòng (do nhân bản ngẫu nhiên từ tệp thô).
- **Số ô khuyết ở cột Tuoi:** `9` ô (trước khi thực hiện điền khuyết).
- **Số ô khuyết ở cột Diem_So:** `8` ô (trước khi thực hiện điền khuyết).

---
### **Câu hỏi 2:** Giải thích tại sao cột Tuoi ưu tiên dùng Trung vị (Median) thay vì Trung bình (Mean) khi tồn tại các giá trị ngoại lai −12 và 195?
**Trả lời:**
1. **Đặc tính của Trung bình (Mean):** Giá trị trung bình phụ thuộc vào tổng đại số của mọi quan sát, do đó cực kỳ nhạy cảm với các giá trị ngoại lai (outliers) cực đoan. Khi cột Tuổi có giá trị âm (`-12`) và giá trị phi thực tế quá lớn (`195`), giá trị Mean bị kéo lệch đáng kể lên trên, không còn phản ánh đúng độ tuổi thực tế của đại đa số sinh viên.
2. **Ưu điểm của Trung vị (Median):** Trung vị là giá trị chia đôi tập dữ liệu đã sắp xếp, thuộc nhóm **thống kê bền vững (Robust Statistics)**. Giá trị Median hoàn toàn không bị ảnh hưởng bởi độ lớn biên cực đoan của `-12` hay `195` (với dữ liệu của em, `Median = 25.0 tuổi`).
3. **Kết luận:** Sử dụng Median để điền khuyết giúp giữ nguyên phân phối độ tuổi tự nhiên, bảo toàn tính đại diện của nhóm sinh viên trước khi đưa vào các thuật toán khai phá dữ liệu.

---
### **Câu hỏi 3:** Tệp dữ liệu sạch cuối cùng còn lại chính xác bao nhiêu dòng?
**Trả lời:**
- Tệp dữ liệu sạch cuối cùng (`2251010027_clean.csv`) còn lại chính xác **116 dòng**.
- **Giải trình biến động số lượng dòng:**
  - Khởi tạo ban đầu: `126 dòng` (gồm 120 dòng gốc + 6 dòng nhân bản trùng lặp).
  - Sau bước loại bỏ trùng lặp (`drop_duplicates`): còn lại **120 dòng**.
  - Sau bước lọc logic (`18 <= Tuoi <= 30` và `0 <= Diem_So <= 10`): loại bỏ đúng **4 dòng vi phạm** (2 dòng ngoại lai tuổi: `-12` và `195`; 2 dòng lỗi điểm số: `-2.0` và `95.0`), còn lại chính xác **116 dòng** sạch hoàn toàn.